In [1]:
import torch
#import torchvision
from torchvision import models
import torchvision.transforms as transforms
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset
from torch.linalg import norm
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from torch.nn.functional import pad
#from torch.func import jacrev
import random
import hickle
import numpy as np
from copy import deepcopy

from trained_models.MNIST.model4 import trainer as t
from utils import trainer as tr
#from sklearn.model_selection import train_test_split
#from utils.rfm_gcnn import agop_gcnn as agc
from utils.rfm_gcnn import recursive_feature_machine as rfm
from utils.rfm_gcnn import utils as ut
#from utils.groupy.gconv.pytorch_gconv.splitgconv2d import P4ConvZ2, P4ConvP4, P4MConvZ2, P4MConvP4M
#from groupy.gconv.make_gconv_indices import *

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
#device='cpu'
print(f"Using device: {device}")


Using device: cuda:0


In [3]:
torch.cuda.empty_cache()

In [4]:
trainloader, valloader, testloader, init_net, net = t.train_net(force_train=False)
#trainloaderv, valloaderv, testloaderv = t.get_loaders_vect(10000,5000)

Model weights loaded successfully.


In [5]:
#tr.visualize_predictions(net, testloader, range(10), device, num_images=36)
print(tr.get_acc_ce(net.to(device), testloader))

99.25


In [5]:
data_batch, labels_batch = next(iter(trainloader))
print(data_batch.shape)
img= data_batch[:10]
#img=img.unsqueeze(0)
img=img.cuda()
print(img.shape)

torch.Size([64, 1, 28, 28])
torch.Size([10, 1, 28, 28])


In [6]:
layer = net.features[0]


In [6]:
M = hickle.load('weight/M_fc_rfm_1.h')
res= []
for i in range(4):
    res.append(ut.nfmvrfm(img, layer, M, pose=i))
res=torch.stack(res, dim=1)
print(res.shape)

NameError: name 'layer' is not defined

In [8]:
#pwd
ut.vis(res, pwd='/work/DLR/experiments/MNIST/agop_gcnn_correlation', fname ="layer0")

Visualization saved to /work/DLR/experiments/MNIST/agop_gcnn_correlation/images/layer0


# RFM GCNN: Debug

In [13]:
layer = net.features[0]

model = GaussRFM(bandwidth=5.0, layer=layer, num_classes=10, diag=False, device=device, mem_gb=3, centering=True, reg=1e-3,              
                 iters=10, p_batch_size=None, bandwidth_mode='constant')



Ms, mses = model.fit(trainloaderv, testloaderv, iters=2, method='lststq', 
            classification=True, verbose=True, M_batch_size=100, 
            class_weight=None, return_best_params=True, bs=None, 
            return_Ms= True, lr_scale=1, total_points_to_sample=100, 
            solver='solve', fit_last_M= True, prefit_eigenpro=True, epochs=3)



Loaders provided
Round 0, Test Acc: 93.40%
Round 0, Test MSE: 0.0314
Sampling AGOP on maximum of 200 total points


Updating M: 2it [01:05, 32.52s/it]


M update complete.
Round 1, Test Acc: 93.50%
Round 1, Test MSE: 0.0321
Sampling AGOP on maximum of 200 total points


Updating M: 2it [01:14, 37.29s/it]


M update complete.
Final MSE: 0.0334
Final Test Acc: 93.20%
Returning best parameters with value: 0.9350
Sampling AGOP on maximum of 200 total points


Updating M: 2it [01:14, 37.25s/it]

M update complete.


In [14]:
hickle.dump(Ms[0], 'weight/M_fc_rfm_0.h')
hickle.dump(Ms[1], 'weight/M_fc_rfm_1.h')
hickle.dump(Ms[2], 'weight/M_fc_rfm_2.h')
#hickle.dump(Ms[3], 'weight/M_fc_rfm_3.h')


/usr/local/lib/python3.10/dist-packages/hickle/lookup.py:1491: SerializedWarning: 'Tensor' type not understood, data is serialized:
  warnings.warn(


In [137]:
'''Helper functions.'''
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as distributions
from torch.nn.functional import pad
from torch.linalg import norm, svd

import os
import numpy as np
import cvxpy as cp
from scipy.linalg import sqrtm, fractional_matrix_power
from matplotlib import pyplot as plt
import math
from copy import deepcopy
from einops import rearrange


#from groupy.gconv.make_gconv_indices import *

plt.switch_backend('Agg')

def float_x(data):
    '''Set data array precision.'''
    return np.float32(data)


################################ MATRIX HELPER FUNCTIONS ###############################################
'''
def matrix_power(M, power):
    """
    Compute the power of a matrix.
    :param M: Matrix to power.
    :param power: Power to raise the matrix to.
    :return: Matrix raised to the power - M^{power}.
    """
    if len(M.shape) == 2:
        assert M.shape[0] == M.shape[1], "Matrix must be square"
        M_cpu = M.cpu()
        original_device = M.device
        try:
            # gpu square root
            S, U = torch.linalg.eigh(M)
            S[S<0] = 0.
            return U @ torch.diag(S**power) @ U.T
        except:
            # stable cpu square root
            M_cpu.diagonal().add_(1e-8)
            if power == 0.5:
                sqrtM = sqrtm(M_cpu)
            else:
                sqrtM = fractional_matrix_power(M_cpu, power)
            sqrtM = torch.from_numpy(sqrtM).to(original_device)
            return sqrtM
    elif len(M.shape) == 1:
        assert M.shape[0] > 0, "Vector must be non-empty"
        M[M<0] = 0.
        return M**power
    else:
        raise ValueError(f"Invalid matrix shape for square root: {M.shape}")

'''

def matrix_power_eigendecomposition(G, alpha, return_type=0):   
    #TODO: Check if fractional_matrix_power from scipy can be used
    eigenvalues, eigenvectors = torch.linalg.eigh(G)    
    # Raise the eigenvalues to the power alpha
    # Clamp to zero to handle potential small negative eigenvalues due to
    # floating-point inaccuracies, which would result in NaN for fractional powers.
    powered_eigenvalues = torch.clamp_min(eigenvalues, 0).pow(alpha)
    V = eigenvectors
    Lambda_powered = torch.diag(powered_eigenvalues)
    powered_matrix = V @ Lambda_powered @ V.T
    if(return_type==1):
        powered_matrix = Lambda_powered @ V.T
    else:
        powered_matrix = V @ Lambda_powered @ V.T
    
    return powered_matrix


def sqrt(G):
    #TODO: Check if sqrtm from scipy can be used
    U, s, Vt = svd(G)
    s = torch.pow(s, 1./2)
    G = U @ torch.diag(s) @ Vt
    return G

def correlation(M, G):
    A = M.clone()
    B = G.clone()
    A -= A.mean()
    B -= B.mean()
    A = A.double()
    B = B.double()
    normM = norm(A.flatten())
    normG = norm(B.flatten())

    corr = torch.dot(A.flatten(), B.flatten()) / (normM * normG)
    return corr
    

def min_max(M):
    return (M - M.min()) / (M.max() - M.min())


################################ IMAGE HELPER FUNCTIONS ###############################################

def patchify(x, layer, pad_type='zeros'):
    '''
        Given an input image (bs,c,h,w) generate (bs,h_out,w_out,c,q,s) respecting stride,padding, 
        w_out is number of pathces along the width for the given stride after padding
        h_out is number of pathces along the height for the given stride after padding
        (q,s) is the kernel dimensions 
    '''
    #TODO: Compare with the padding done in gcnn. Ensure they are same.
    #TODO: double check width height order
    
    input_shape = x.size()
    in_channels = layer.in_channels
    ip_stab = layer.input_stabilizer_size
    patch_size = layer.kernel_size 
    stride_size = layer.stride
    padding = layer.padding 
    
    x = x.reshape(input_shape[0], in_channels*ip_stab, input_shape[-2], input_shape[-1])
    q1, q2 = patch_size
    s1, s2 = stride_size
    if padding is None:
        pad_1 = (q1-1)//2
        pad_2 = (q2-1)//2
    else:
        pad_1, pad_2 = padding

    pad_dims = (pad_2, pad_2, pad_1, pad_1)
    if pad_type == 'zeros':
        x = pad(x, pad_dims)
    elif pad_type == 'circular':
        x = pad(x, pad_dims, 'circular')
        
    patches = x.unfold(2, q1, s1).unfold(3, q2, s2) #(bs, c, h_out, w_out, q, s)
    #print("Image Shape1",patches.shape)
    patches = patches.transpose(1, 3).transpose(1, 2) #(bs, h_out, w_out, c, q, s) 
    #print("Image Shape2",patches.shape)
    return patches

'''
def expand_image(X, ps=3, pad_mode="circular"):
    """
    X : (n, c, p, q)
    out : (n, c, p*ps, q*ps)
    """

    n, c, p, q = X.shape

    pad_sz = ps//2
    if pad_mode=="zero":
        pad = (pad_sz,pad_sz,pad_sz,pad_sz)
        X_patched = F.pad(X, pad)
    elif pad_mode=="circular":
        X_patched = torch.from_numpy(np.pad(X, ((0,0),(0,0),(pad_sz,pad_sz),(pad_sz,pad_sz)), mode='wrap'))

    X_patched = X_patched.unfold(2,ps,1).unfold(3,ps,1) # (n, c, p, q, ps, ps)
    X_patched = X_patched.transpose(-2,-3) # (n, c, p, ps, q, ps)
    X_expanded = X_patched.reshape(n,c,p*ps,q*ps)
    return X_expanded
'''

def reduce_image(X, depth, ps=3):
    """
    X : (n, c, p*ps, q*ps)
    out : (n, c, p, q)
    """
    n, c, P, Q = X.shape
    p = P//ps
    q = Q//ps

    X = X.reshape(n, c, p, ps, q, ps)
    if depth == 0:
        return X.norm(dim=(3,5))
    else:
        X = X.norm(dim=(3,5))
        #X = torch.max(X, dim=1)[0]
        return X
        #X = X**2 
        #X = X.sum(dim=(1,3,5))
        #return X.sqrt()

    #X = torch.permute(X, (0, 1, 3, 5, 2, 4))
    #X = X.reshape(n, c*ps*ps, p*q)
    #pad_sz = ps//2
    #folded = fold(X, output_size=(p, q), kernel_size=(ps, ps), padding=(pad_sz, pad_sz))

    #ones = torch.ones(X.shape)
    #ones = fold(ones, output_size=(p, q), kernel_size=(ps, ps), padding=(pad_sz, pad_sz))
    #return folded/ones



def multiply_patches(X, M, ps=3):
    """
    Applies the covariance matrix M
    X : (n, c, p*ps, q*ps)
    M_ : (c*w*h, c*w*h)
    out : (n, c, p*ps, q*ps)
    """
    #n = X.shape[0]
    chunk_size = 5000
    #leftover_bool = int(n%chunk_size>0)
    #batches = np.array_split(np.arange(n), n//chunk + leftover_bool)
    batches = torch.split(X, chunk_size, dim=0)

    #M = sqrt(M)
    M = matrix_power_eigendecomposition(M, 0.5, return_type=0)

    Xs = []
    for Xb in batches:
        m, c, P, Q = Xb.shape
        p = P//ps
        q = Q//ps
        #Xb = rearrange(Xb, 'm c (p h) (q w) -> (m p q) (c h w)', p=p, q=q, w=ps, h=ps)
        Xb = rearrange(Xb, 'm c (p h) (q w) -> (c h w) (m p q)', p=p, q=q, w=ps, h=ps)
        #TODO: Check the datatypes of Xb and M.
        #Xb = Xb @ M
        Xb = M @ Xb
        #Xb = rearrange(Xb, '(m p q) (c w h) -> m c (p w) (q h)', m=m, p=p, q=q, c=c, w=ps, h=ps)
        #Xb.relu_()
        Xb = rearrange(Xb, '(c h w) (m p q) -> m c (p h) (q w)', m=m, p=p, q=q, c=c, w=ps, h=ps) 
        Xs.append(Xb)
    return torch.cat(Xs, dim=0)




def vis(tensor_input, pwd, fname, title="Feature Analysis", 
                  apply_relu=False, reduction='mean'):
    """
    Uses PyTorch for ReLU and Reduction.
    reduction='max' now uses standard torch.max (picks the most positive value).
    """
    # 1. ReLU Option (In-place on a clone to protect original data)
    data = tensor_input.clone()
    if apply_relu:
        data = torch.relu(data)
    
    # 2. Channel Reduction in Torch
    # data is (N, P, 2, C, H, W) -> we reduce dim 3 (C)
    if reduction == 'max':
        # torch.max returns a tuple (values, indices); we take [0]
        data_collapsed = torch.max(data, dim=3, keepdim=True)[0]
    else:
        data_collapsed = torch.mean(data, dim=3, keepdim=True)

    # 3. Move to CPU/Numpy only for Plotting
    data_np = data_collapsed.detach().cpu().numpy()
    N, P, _, _, H, W = data_np.shape

    # 4. Setup Plotting
    rows, cols = N, P * 2
    save_path = os.path.join(pwd, 'images')
    os.makedirs(save_path, exist_ok=True)

    # If ReLU is on, 'magma' or 'viridis' is best. 
    # If ReLU is off, 'RdBu_r' is better to see those negatives.
    #COLOR_MAP = 'hot' if apply_relu else 'RdBu_r'
    COLOR_MAP = 'hot'
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5), squeeze=False)
    conditions = [("Backprop", 0), ("No Backprop", 1)]

    # 5. Independent Scaling per Condition
    for _, d_idx in conditions:
        cond_data = data_np[:, :, d_idx, :]
        v_min, v_max = cond_data.min(), cond_data.max()
        
        # Ensure v_min and v_max are not identical to avoid Matplotlib warnings
        if v_min == v_max:
            v_max += 1e-6

        for n in range(N):
            for p in range(P):
                col_idx = (p * 2) + d_idx
                ax = axes[n, col_idx]
                
                img = data_np[n, p, d_idx, 0]
                im = ax.imshow(img, cmap=COLOR_MAP, vmin=v_min, vmax=v_max)
                ax.axis('off')
                
                # Labels
                if p == 0 and d_idx == 0:
                    ax.set_ylabel(f"Sample {n+1}", rotation=90, size=12, weight='bold', labelpad=20)
                if n == 0:
                    ax.set_title(f"{conditions[d_idx][0]}", fontsize=10)

    # 6. Formatting
    plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.93])
    fig.suptitle(f"{title}\n(Torch {reduction.upper()}, ReLU={apply_relu})", fontsize=16, y=0.98)
    
    # Pose Labels
    for p in range(P):
        ax_left = axes[0, p*2]
        ax_right = axes[0, p*2+1]
        center_x = (ax_left.get_position().x0 + ax_right.get_position().x1) / 2
        fig.text(center_x, 0.94, f"Pose {p+1}", ha='center', weight='bold', fontsize=14)

    plt.savefig(os.path.join(save_path, fname))
    plt.close(fig)


def nfmvrfm(img, layer, S, pose=0):
    #TODO: Handle other poses
    pat = patchify(img, layer)
    pat = pat.transpose(2,3).transpose(1,2).transpose(3,4) #(bs, c, h_out, q, w_out, s)
    pat = pat.reshape(pat.shape[0], pat.shape[1], pat.shape[2]*pat.shape[3], pat.shape[4]*pat.shape[5])
    pat = pat.cuda()
    N =  get_nfm(layer, pose)
    N = N.cuda()
    o1 = multiply_patches(pat, N, ps=layer.ksize)
    #o1 = reduce_image(o1, 0, ps=layer.ksize)

    M = find_covariance_matrix_m(layer, S, pose=pose)
    #M = tup[0]
    #M = torch.from_numpy(tup[0])
    #M = M.float()
    #M = ut.sample_normal(M, layer.weight.shape[0])
    M = M.cuda()
    o2 = multiply_patches(pat, M, ps=layer.ksize)
    #o2 = reduce_image(o2, 0, ps=layer.ksize)
    combined_tensor = torch.stack((o1, o2), dim=1)
    return combined_tensor
    

################################ GCNN HELPER FUNCTIONS ###############################################

def trans_filter(w, inds):
    
    #TODO: Reference this function directly from the SplitConv2d class of gcnn.
    inds_reshape = inds.reshape((-1, inds.shape[-1])).astype(np.int64)
    w_indexed = w[:, :, inds_reshape[:, 0].tolist(), inds_reshape[:, 1].tolist(), inds_reshape[:, 2].tolist()]
    w_indexed = w_indexed.reshape(w_indexed.size()[0], w_indexed.size()[1],
                                    inds.shape[0], inds.shape[1], inds.shape[2], inds.shape[3])
    w_transformed = w_indexed.permute(0, 2, 1, 3, 4, 5)
    return w_transformed.contiguous()

def get_nfm(layer, pose=None):
    
    # Extract all the meta info of the current conv layer.
    (q, s) = layer.kernel_size
    (pad1, pad2) = layer.padding
    (s1, s2) = layer.stride 
    in_channels = layer.in_channels
    input_stabilizer_size = layer.input_stabilizer_size
    tw = trans_filter(layer.weight, layer.inds)
    
    
    if pose is not None:      
        tw = tw[:, pose, :, :, :, :]
        tw_shape = (layer.out_channels, layer.in_channels * layer.input_stabilizer_size,
                            layer.ksize, layer.ksize)
        W = tw.reshape(tw_shape)
    else:     
        tw = trans_filter(layer.weight, layer.inds)   
        tw_shape = (layer.out_channels * layer.output_stabilizer_size,
                            layer.in_channels * layer.input_stabilizer_size,
                            layer.ksize, layer.ksize)
        W = tw.reshape(tw_shape)
        
    k, ki, q, s= W.shape
    
    # Reshape W which into a (k, c*q*s) matrix. 
    W = W.reshape(-1, ki*q*s)
                
    # Compute WtW which is (c*q*s,c*q*s) matrix
    M = torch.einsum('nd, nD -> dD', W, W)
    return M

def get_permutation_matrix(v1, v2):
    
    if v1.shape != v2.shape or v1.ndim != 1:
        raise ValueError("v1 and v2 must be 1D tensors of the same shape.")

    n = v1.size(0)
    perm_indices = torch.empty_like(v2, dtype=torch.long)
    for i in range(n):
        perm_indices[i] = (v1 == v2[i]).nonzero(as_tuple=True)[0][0]
   
    P = F.one_hot(perm_indices.long(), num_classes=n).float()

    return P

def sample_normal(covariance_matrix_M, num_rows_k):
   
    # Create a mean vector of zeros, compatible with the dimensions of M
    device = covariance_matrix_M.device
    dtype = covariance_matrix_M.dtype
    matrix_dim_t = covariance_matrix_M.shape[0]
    #mean_vector = torch.zeros(matrix_dim_t, dtype=dtype)
    mean_vector = torch.zeros((matrix_dim_t,), dtype=dtype, device=device)
    #mean_vector.to(device)
    try:
        mvn = distributions.MultivariateNormal(loc=mean_vector, covariance_matrix=covariance_matrix_M)
    except Exception as e:
        print(f"Error creating MultivariateNormal distribution: {e}")
        print("Please ensure your covariance_matrix_M is symmetric positive definite.")
        # Attempt to make it PSD for robustness in example, by adding a small diagonal perturbation
        # In a real scenario, you'd ensure your input M is correctly formed.
        min_eigval = torch.min(torch.linalg.eigvalsh(covariance_matrix_M)).item()
        if min_eigval <= 0:
            print(f"Warning: Covariance matrix has non-positive eigenvalue ({min_eigval:.2e}). Adding jitter.")
            covariance_matrix_M = covariance_matrix_M + torch.eye(matrix_dim_t, device=device, dtype=dtype) * (abs(min_eigval) + 1e-6)
            mvn = distributions.MultivariateNormal(loc=mean_vector, covariance_matrix=covariance_matrix_M)


    # Sample num_rows_k times from the distribution
    # The .sample() method will return a tensor of shape (num_rows_k, matrix_dim_t)
    sampled_matrix = mvn.sample((num_rows_k,))

    return sampled_matrix

def find_M(S, N):
    S_scaled = S / 4
    
    # Calculate the Frobenius inner product: trace(S_scaled.T @ N)
    inner_product = torch.sum(S_scaled * N) 
    epsilon = torch.sum(S_scaled * S_scaled) / inner_product  
    M = S_scaled - epsilon * N
    
    # Check PSD condition
    eigenvalues = torch.linalg.eigvalsh(M)
    print("eigen",eigenvalues)
    if torch.all(eigenvalues >= -1e-6):
        return M
    else:
        print(f"Calculated epsilon {epsilon:.4f} makes M non-PSD.")
        return M



def find_M1(S, N, P, start_epsilon=100.0, diff_threshold=0.4):
    epsilon = start_epsilon
    S_scaled = S / 4
    flag= 0
    
    while epsilon > 0.0001:  # Lower floor to allow more search range
        M = S_scaled - epsilon * N
        #M = epsilon * N
        
        # 1. Check if it's PSD
        eigenvalues = torch.linalg.eigvalsh(M)
        is_psd = torch.all(eigenvalues >= -1e-6)
        
        if is_psd:
            flag =1
            A = M
            B = P @ M @ P.T
            sim = correlation(A, B)           
            if sim <= diff_threshold:
                print("epsilon",epsilon)
                return M
            else:
                # If we are here, epsilon is too small to meet the difference req
                print(f"Failed: Cosine sim {sim:.4f} above threshold {diff_threshold}")
                #return None
        
        epsilon -= 0.1 
    if(flag==0):
        print("Error: Could not find a PSD matrix within epsilon range")
    return None

def find_covariance_matrix_m(layer, S, pose=0, solver_name='SCS'):

    dummy= deepcopy(layer)
    dummy.out_channels = 1
    temp = torch.arange(dummy.in_channels*dummy.input_stabilizer_size*dummy.ksize*dummy.ksize, dtype=torch.float)
    temp= temp.reshape(dummy.out_channels,dummy.in_channels,dummy.input_stabilizer_size,dummy.ksize,dummy.ksize)
    dummy.weight = nn.Parameter(temp)
    dummy.inds = dummy.make_transformation_indices()
    tw = trans_filter(dummy.weight, dummy.inds)
    tw_shape = (dummy.out_channels * dummy.output_stabilizer_size,
                        dummy.in_channels * dummy.input_stabilizer_size,
                        dummy.ksize, dummy.ksize)
    tw = tw.reshape(tw_shape)
    tw_test= tw.reshape(dummy.out_channels, dummy.output_stabilizer_size,
                        dummy.in_channels , dummy.input_stabilizer_size,
                        dummy.ksize, dummy.ksize)
    
    print("tw_shape",tw_test.shape)

    
    v1 = tw_test[0][0].reshape(-1) 
    v2 = tw_test[0][1].reshape(-1) 
    P = get_permutation_matrix(v1, v2)
    print(P.shape)
    # Verification: Check that P @ v1 gives v2
    v1_col = v1.unsqueeze(1)
    v2_calc = P @ v1_col
    v2_calc= v2_calc.squeeze(1)
    print("\nVerification (P @ v1):\n", v2_calc)
    print("\nAre they equal? (within tolerance):", torch.allclose(v2_calc, v2))
    
    torch.manual_seed(4000)
   
    for i in range(10):
        T = 0
        #R = torch.randn_like(S)
        dist = torch.distributions.Cauchy(torch.tensor([0.0]), torch.tensor([1.0]))
        R = dist.sample(S.shape).squeeze()
        for i in range(tw_test.shape[1]):
            P_i = torch.linalg.matrix_power(P, i)
            T += P_i @ R @ (P_i).T
        N = R - T/4 # Note that Q(N)=0.Thus, N is non zero and non invariant to the cyclic group
        print("N matrix",N)
        #M = S/4 + epsilon*N # This is a non invariant solution to the equation
        #M = find_M2(S, N, P)
        M = find_M1(S, N, P)
        if(M is not None):
            break

    P_i = torch.linalg.matrix_power(P, pose)
    return P_i @ M @ (P_i).T
    

    




    '''
    
    np.random.seed(5000)
    M_dim = S.shape[0]
    #M_rand = np.random.rand(M_dim, M_dim)
    #M_guess = M_rand.T @ M_rand  
    M = cp.Variable((M_dim, M_dim), symmetric=True)
    #M.value = M_guess
    sum_term = 0
    for P in P_matrices:
        sum_term += P @ M @ P.T
    
    sum_term += M
    lambda_init = 1.0
    
    objective = cp.Minimize(
        cp.norm(sum_term - S, 'fro') + 
        lambda_init * cp.sum_squares(M-M_guess)
    )
    
    
    objective = cp.Minimize(
        cp.norm(sum_term - S, 'fro')
    )
    orthogonality_constraint = cp.sum(cp.multiply(M, S/4)) == 0
    constraints = [
        M >> 0,
        orthogonality_constraint
    ]
    
      
    problem = cp.Problem(objective, constraints)
    try:
        problem.solve(solver=solver_name)
    except Exception as e:
        print(f"An unexpected error occurred during solving: {e}")
        return None, None, "Error"
    if problem.status in [cp.OPTIMAL]:
         M = M.value 
         #print(M)
         M = torch.from_numpy(M)
         M = M.float()
         return P_matrices[pose]@ M @ P_matrices[pose].T, problem.value, problem.status
    else:
        print(f"Problem did not solve to optimality. Status: {problem.status}")
        print("This could mean the problem is infeasible, unbounded, or the solver failed to converge.")
        return None, problem.value, problem.status
'''

################################ LOADER HELPER FUNCTIONS ###############################################

def get_data_from_loader(data_loader, layer, num_classes):
    """
    Get data from a data loader.
    :param data_loader: Torch DataLoader to get data from.
    :return: Tuple of tensors - (X, y).
    """
    X, y = [], []
    
    for idx, batch in enumerate(data_loader):
        inputs, labels = batch
        inputs = patchify(inputs, layer) 
        inputs = inputs.reshape(inputs.shape[0], inputs.shape[1], inputs.shape[2], -1)
        X.append(inputs)
        y.append(F.one_hot(labels, num_classes).to(torch.float32))
    return torch.cat(X, dim=0), torch.cat(y, dim=0)


In [119]:
#M = hickle.load('weight/M_fc_rfm_1.h')
M = get_nfm(layer).detach()
print(M.dtype)
pose=1
o = get_nfm(layer, pose=0)
#o = find_covariance_matrix_m(layer, M, pose=0)
print(o)
N = get_nfm(layer, pose=2)
#N = find_covariance_matrix_m(layer, M, pose=1)
print(N)
o= o.cuda()
N= N.cuda()
print(correlation(N, o))



torch.float32
tensor([[ 0.6653, -0.1011, -0.2787,  0.2669,  0.0798, -0.2954,  0.3949,  0.0213,
         -0.4567],
        [-0.1011,  0.5940,  0.0635,  0.1222,  0.1842, -0.2744,  0.2324, -0.0070,
          0.1346],
        [-0.2787,  0.0635,  0.3486, -0.0983,  0.0142,  0.2763, -0.2180, -0.1859,
          0.2068],
        [ 0.2669,  0.1222, -0.0983,  0.6888,  0.0504, -0.4739,  0.4182, -0.0370,
         -0.1369],
        [ 0.0798,  0.1842,  0.0142,  0.0504,  0.8386, -0.1636,  0.1079,  0.1473,
          0.1620],
        [-0.2954, -0.2744,  0.2763, -0.4739, -0.1636,  0.8752, -0.5754,  0.1322,
          0.2694],
        [ 0.3949,  0.2324, -0.2180,  0.4182,  0.1079, -0.5754,  0.7980,  0.1653,
         -0.3474],
        [ 0.0213, -0.0070, -0.1859, -0.0370,  0.1473,  0.1322,  0.1653,  0.6477,
          0.1097],
        [-0.4567,  0.1346,  0.2068, -0.1369,  0.1620,  0.2694, -0.3474,  0.1097,
          0.6730]], grad_fn=<ViewBackward0>)
tensor([[ 0.6730,  0.1097, -0.3474,  0.2694,  0.1620, -0.136

In [138]:
#M = hickle.load('weight/M_fc_rfm_1.h')
M = get_nfm(layer).detach().cpu()
res= []
for i in range(4):
    res.append(nfmvrfm(img, layer, M, pose=i))
res=torch.stack(res, dim=1)
print(res.shape)
vis(res, apply_relu=True, pwd='/work/DLR/experiments/MNIST/agop_gcnn_correlation', fname ="layer0", reduction='mean')

tw_shape torch.Size([1, 4, 1, 1, 3, 3])
torch.Size([9, 9])

Verification (P @ v1):
 tensor([2., 5., 8., 1., 4., 7., 0., 3., 6.], grad_fn=<SqueezeBackward1>)

Are they equal? (within tolerance): True
N matrix tensor([[-1.5392e+00, -5.1538e+00, -1.3169e+00,  1.4120e+00,  1.3201e-04,
          1.9333e-01, -1.5190e-01, -1.6705e+00,  1.7538e-01],
        [-4.1921e+00,  1.1758e+00,  6.5887e-01, -1.6243e-01, -2.0163e+00,
          2.2881e+00,  2.3116e+00, -5.3824e+00, -1.1179e+00],
        [-3.0094e-01,  1.1139e+00, -7.8186e-01,  5.7741e-01,  2.3135e+00,
          1.7800e+00, -1.6186e+00, -2.1550e+00,  9.0216e-01],
        [-6.3937e-01, -2.6454e+00,  2.5978e+00,  1.0193e+00, -1.5611e+00,
         -7.4915e+00,  1.3552e+00,  5.5206e-01, -3.0937e+00],
        [ 9.2437e-02,  2.4418e-01,  1.9367e+00, -4.8144e-01,  0.0000e+00,
          8.3124e-01,  6.3400e-01, -5.9398e-01, -2.6632e+00],
        [-8.7426e-03, -3.9961e-02,  7.7479e-01,  1.8879e+01,  4.5611e+00,
         -1.1831e+00, -2.6302e+00, -1.

TypeError: unsupported operand type(s) for @: 'Tensor' and 'NoneType'

Visualization saved to /work/DLR/experiments/MNIST/agop_gcnn_correlation/images/layer0
